In [1]:
%pip install requests beautifulsoup4 pymysql

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
import requests
from bs4 import BeautifulSoup

url = "http://www.bspider.top/anjuke/"
r = requests.get(url, headers={"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/156.0.0.0 Safari/537.36 Edg/156.0.0.0"}, timeout=30)
r.raise_for_status()
r.encoding = r.apparent_encoding
soup = BeautifulSoup(r.text, "html.parser")

In [3]:
rows = soup.select("div.property")
print(len(rows))

60


In [10]:
row = rows[0]
node = row.select_one("h3")

In [11]:
if node:
    title = node.get_text(strip=True)
    print(title)
else:
    title = ""
    print("没有找到h3标题")

万科物业自营 南坪六公里金域翰林 三号线精装正三房带家具电器


In [12]:
row = rows[0]

# 楼盘名称
title_node = row.select_one("h3")
title = title_node.get_text(strip=True) if title_node else ""

# 地址
addr_node = row.select_one("p.property-content-info-comm-address")
address = addr_node.get_text(strip=True) if addr_node else ""

# 取出所有信息p标签，按顺序：户型、面积、朝向、楼层、建造年代
info_list = row.select("p.property-content-info-text")

house_type = area = direction = floor = build_year = ""
if len(info_list) >= 1:
    house_type = info_list[0].get_text(strip=True)
if len(info_list) >= 2:
    area = info_list[1].get_text(strip=True)
if len(info_list) >= 3:
    direction = info_list[2].get_text(strip=True)
if len(info_list) >= 4:
    floor = info_list[3].get_text(strip=True)
if len(info_list) >= 5:
    build_year = info_list[4].get_text(strip=True)

# 总价
total_price_node = row.select_one("p.property-price-total")
total_price = total_price_node.get_text(strip=True) if total_price_node else ""

# 单价
avg_price_node = row.select_one("p.property-price-average")
avg_price = avg_price_node.get_text(strip=True) if avg_price_node else ""

print("楼盘：", title)
print("地址：", address)
print("户型：", house_type)
print("面积：", area)
print("朝向：", direction)
print("楼层：", floor)
print("建造年代：", build_year)
print("总价：", total_price)
print("单价：", avg_price)

楼盘： 万科物业自营 南坪六公里金域翰林 三号线精装正三房带家具电器
地址： 南岸学府大道二塘路59号
户型： 3室2厅1卫
面积： 89.9㎡
朝向： 南北
楼层： 低层(共27层)
建造年代： 2019年建造
总价： 148万
单价： 16465元/㎡


In [13]:
from bs4 import BeautifulSoup

# soup = BeautifulSoup(html, 'html.parser')
# rows = soup.select("房源卡片外层选择器")
row = rows[0]   # 取第一条房源

# -------- select_one 用法 --------
title_node = row.select_one("h3")
title = title_node.get_text(strip=True) if title_node else ""

address_node = row.select_one("p.property-content-info-comm-address")
address = address_node.get_text(strip=True) if address_node else ""

total_price_node = row.select_one("p.property-price-total")
total_price = total_price_node.get_text(strip=True) if total_price_node else ""

avg_price_node = row.select_one("p.property-price-average") # 单价定位
avg_price = avg_price_node.get_text(strip=True) if avg_price_node else ""

# -------- select 用法 --------
info_nodes = row.select("p.property-content-info-text")

house_type = area = direction = floor = build_year = ""
if len(info_nodes) >= 1:
    house_type = info_nodes[0].get_text(strip=True)
if len(info_nodes) >= 2:
    area = info_nodes[1].get_text(strip=True)    # 面积取值
if len(info_nodes) >= 3:
    direction = info_nodes[2].get_text(strip=True)
if len(info_nodes) >= 4:
    floor = info_nodes[3].get_text(strip=True)
if len(info_nodes) >= 5:
    build_year = info_nodes[4].get_text(strip=True) #建造年代

# 输出8个字段
print("楼盘：", title)
print("地址：", address)
print("户型：", house_type)
print("面积：", area)
print("朝向：", direction)
print("楼层：", floor)
print("建造年代：", build_year)
print("单价：", avg_price)

楼盘： 万科物业自营 南坪六公里金域翰林 三号线精装正三房带家具电器
地址： 南岸学府大道二塘路59号
户型： 3室2厅1卫
面积： 89.9㎡
朝向： 南北
楼层： 低层(共27层)
建造年代： 2019年建造
单价： 16465元/㎡


In [4]:
import csv
from bs4 import BeautifulSoup

# 字段表头
headers = ["楼盘标题","地址","户型","面积","朝向","楼层","建造年代","单价"]
all_data = []

# 循环每一条房源卡片rows
for row in rows:
    title_node = row.select_one("h3")
    title = title_node.get_text(strip=True) if title_node else ""

    address_node = row.select_one("p.property-content-info-comm-address")
    address = address_node.get_text(strip=True) if address_node else ""

    avg_price_node = row.select_one("p.property-price-average")
    avg_price = avg_price_node.get_text(strip=True) if avg_price_node else ""

    info_nodes = row.select("p.property-content-info-text")
    house_type = area = direction = floor = build_year = ""
    if len(info_nodes) >= 1:
        house_type = info_nodes[0].get_text(strip=True)
    if len(info_nodes) >= 2:
        area = info_nodes[1].get_text(strip=True)
    if len(info_nodes) >= 3:
        direction = info_nodes[2].get_text(strip=True)
    if len(info_nodes) >= 4:
        floor = info_nodes[3].get_text(strip=True)
    if len(info_nodes) >= 5:
        build_year = info_nodes[4].get_text(strip=True)

    line = [title,address,house_type,area,direction,floor,build_year,avg_price]
    all_data.append(line)

# 写入csv
with open("houseinfo.csv","w",encoding="utf-8-sig",newline="") as f:
    writer = csv.writer(f)
    writer.writerow(headers)
    writer.writerows(all_data)

In [5]:
from urllib.parse import urljoin

current_url = "http://www.bspider.top/anjuke/"

next_node = soup.select_one("a.next.next-active")
if next_node:
    btn_text = next_node.get_text(strip=True)   
    href = next_node.get("href")                
    next_url = urljoin(current_url, href)       
    print("按钮文字：", btn_text)
    print("href 原值：", href)
    print("urljoin 补全后：", next_url)
else:
    print("当前页没有活动的下一页按钮（可能已是末页）")

按钮文字： 下一页
href 原值： ?page=2
urljoin 补全后： http://www.bspider.top/anjuke/?page=2


In [6]:
def parse_property(row):
    """解析一条房源卡片，按固定顺序返回8个字段：
    title → house → address → struct → area → floor → maketime → price"""
    # 标题：h3 是房源的卖点标题
    node = row.select_one("h3")
    title = node.get_text(strip=True) if node else ""

    # 楼盘：p.property-content-info-comm-name
    node = row.select_one("p.property-content-info-comm-name")
    house = node.get_text(strip=True) if node else ""

    # 地址：p.property-content-info-comm-address
    node = row.select_one("p.property-content-info-comm-address")
    address = node.get_text(strip=True) if node else ""

    # 属性组 p.property-content-info-text 按顺序是：户型、面积、朝向、楼层、建造年代
    info_list = row.select("p.property-content-info-text")
    struct = info_list[0].get_text(strip=True) if len(info_list) >= 1 else ""
    area = info_list[1].get_text(strip=True) if len(info_list) >= 2 else ""      # 下标1才是面积（带㎡）
    floor = info_list[3].get_text(strip=True) if len(info_list) >= 4 else ""     # 下标3是楼层，别取成下标2的朝向
    maketime = info_list[4].get_text(strip=True) if len(info_list) >= 5 else ""  # 建造年代可能缺失，缺失填空字符串

    # 单价：p.property-price-average 是元/㎡单价，不是总价
    node = row.select_one("p.property-price-average")
    price = node.get_text(strip=True) if node else ""

    return [title, house, address, struct, area, floor, maketime, price]

# 先解析第一条，逐字段与网页核对
print(parse_property(rows[0]))

['万科物业自营 南坪六公里金域翰林 三号线精装正三房带家具电器', '万科金域学府翰林菁华里', '南岸学府大道二塘路59号', '3室2厅1卫', '89.9㎡', '低层(共27层)', '2019年建造', '16465元/㎡']


In [7]:
import time

# 完整浏览器 UA：简单 UA 会被教学站拦截返回访问验证页
REQUEST_HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/156.0.0.0 Safari/537.36 Edg/156.0.0.0"}

def fetch_page(page_url):
    """请求一个列表页并返回 BeautifulSoup 对象；返回 None 表示失败或遇到访问验证页"""
    try:
        r = requests.get(page_url, headers=REQUEST_HEADERS, timeout=30)
        r.raise_for_status()
        r.encoding = r.apparent_encoding
    except Exception as e:
        print("请求失败：", page_url, "，错误：", e)
        return None
    page_soup = BeautifulSoup(r.text, "html.parser")
    # 即使状态码是200，也可能是访问验证页；房源列表为空也视为异常
    if "访问验证" in page_soup.get_text() or len(page_soup.select("div.property")) == 0:
        print("警告：", page_url, "返回的不是房源列表（可能是访问验证页），停止采集")
        return None
    return page_soup

# ---- 先试采2页，确认第二页没有重复第一页 ----
results = []
page_url = current_url
for page_no in range(1, 3):
    page_soup = fetch_page(page_url)
    if page_soup is None:
        break
    page_rows = page_soup.select("div.property")
    first_rec = parse_property(page_rows[0])   # 本页第一条，用来对比两页是否重复
    for row in page_rows:
        results.append(parse_property(row))
    print(f"第{page_no}页 | {page_url} | 本页 {len(page_rows)} 条 | 累计 {len(results)} 条 | 本页第一条：{first_rec[0][:20]}")
    # 找下一页并补全地址
    next_node = page_soup.select_one("a.next.next-active")
    if next_node:
        page_url = urljoin(page_url, next_node.get("href"))
        time.sleep(1.5)   # 至少等待1秒再请求下一页

print("两页第一条不同即说明没有重复采集")

第1页 | http://www.bspider.top/anjuke/ | 本页 60 条 | 累计 60 条 | 本页第一条：万科物业自营 南坪六公里金域翰林 三号线
第2页 | http://www.bspider.top/anjuke/?page=2 | 本页 60 条 | 累计 120 条 | 本页第一条：万科物业自营 朝中庭 业主诚心出售
两页第一条不同即说明没有重复采集


In [8]:
MAX_PAGES = 30   

def crawl_all(start_url):
    """从 start_url 开始翻页采集全部房源，返回结果列表"""
    all_rows = []      
    visited = set()    
    page_url = start_url
    page_no = 0

    while page_no < MAX_PAGES:
        # 1. 检查地址没有越出教学站，且没有访问过
        if not page_url.startswith("http://www.bspider.top/anjuke/"):
            print("地址越出教学站，中止：", page_url)
            break
        if page_url in visited:
            print("出现重复地址，中止：", page_url)
            break
        visited.add(page_url)
        page_no += 1

        # 2. 请求当前页，检查响应内容
        page_soup = fetch_page(page_url)
        if page_soup is None:
            print(f"第{page_no}页请求或校验失败，中止")
            break

        # 3. 解析本页房源，加入结果列表
        page_rows = page_soup.select("div.property")
        if len(page_rows) == 0:
            print(f"第{page_no}页房源列表为空，结构异常，中止")
            break
        for row in page_rows:
            all_rows.append(parse_property(row))
        print(f"第{page_no}页 | {page_url} | 本页 {len(page_rows)} 条 | 累计 {len(all_rows)} 条")

        # 4. 找下一页，判断停止条件
        next_node = page_soup.select_one("a.next.next-active")
        if next_node is None:
            # 只有分页区域存在且"下一页"按钮已禁用，才算正常到达末页
            if page_soup.select_one("ul.page") is not None:
                print("分页区域存在且下一页按钮已禁用，确认到达末页，采集完成")
            else:
                print("分页区域消失，属于结构异常，本次记为中止而非采集完成")
            break

        # 5. 补全地址，等待至少1秒再请求下一页
        page_url = urljoin(page_url, next_node.get("href"))
        time.sleep(1.5)

    if page_no >= MAX_PAGES:
        print("达到安全页数上限，记为中止，不能算采集完成")
    return all_rows

results = crawl_all("http://www.bspider.top/anjuke/")
print("采集结束，去重前共", len(results), "条")

第1页 | http://www.bspider.top/anjuke/ | 本页 60 条 | 累计 60 条
第2页 | http://www.bspider.top/anjuke/?page=2 | 本页 60 条 | 累计 120 条
第3页 | http://www.bspider.top/anjuke/?page=3 | 本页 60 条 | 累计 180 条
第4页 | http://www.bspider.top/anjuke/?page=4 | 本页 60 条 | 累计 240 条
第5页 | http://www.bspider.top/anjuke/?page=5 | 本页 60 条 | 累计 300 条
第6页 | http://www.bspider.top/anjuke/?page=6 | 本页 60 条 | 累计 360 条
第7页 | http://www.bspider.top/anjuke/?page=7 | 本页 60 条 | 累计 420 条
第8页 | http://www.bspider.top/anjuke/?page=8 | 本页 60 条 | 累计 480 条
第9页 | http://www.bspider.top/anjuke/?page=9 | 本页 60 条 | 累计 540 条
第10页 | http://www.bspider.top/anjuke/?page=10 | 本页 60 条 | 累计 600 条
分页区域存在且下一页按钮已禁用，确认到达末页，采集完成
采集结束，去重前共 600 条


In [9]:
import csv
import os

CSV_HEADERS = ["title", "house", "address", "struct", "area", "floor", "maketime", "price"]

# 按8个字段完全相同去重
before = len(results)
seen = set()
unique_rows = []
for rec in results:
    key = tuple(rec)
    if key not in seen:
        seen.add(key)
        unique_rows.append(rec)
after = len(unique_rows)
print(f"去重前 {before} 条，完全相同的重复记录 {before - after} 条，去重后 {after} 条")

# 检查每条记录恰好8项
for rec in unique_rows:
    if len(rec) != 8:
        raise ValueError(f"记录字段数不是8：{rec}")

# 保存 CSV：编码 utf-8-sig，newline=""
os.makedirs("data", exist_ok=True)
with open("data/houseinfo.csv", "w", encoding="utf-8-sig", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(CSV_HEADERS)
    writer.writerows(unique_rows)
print("已保存 data/houseinfo.csv")

# 用 DictReader 重新读取，检查数据行数（表头不计入）
with open("data/houseinfo.csv", "r", encoding="utf-8-sig", newline="") as f:
    reader = csv.DictReader(f)
    print("表头：", reader.fieldnames)
    csv_rows = list(reader)
print("回读数据行数（不含表头）：", len(csv_rows))
print("第一条：", [csv_rows[0][h] for h in CSV_HEADERS])
print("最后一条：", [csv_rows[-1][h] for h in CSV_HEADERS])

去重前 600 条，完全相同的重复记录 102 条，去重后 498 条
已保存 data/houseinfo.csv
表头： ['title', 'house', 'address', 'struct', 'area', 'floor', 'maketime', 'price']
回读数据行数（不含表头）： 498
第一条： ['万科物业自营 南坪六公里金域翰林 三号线精装正三房带家具电器', '万科金域学府翰林菁华里', '南岸学府大道二塘路59号', '3室2厅1卫', '89.9㎡', '低层(共27层)', '2019年建造', '16465元/㎡']
最后一条： ['恒大照母山 精装3房 内圈位置 户型方正 拎包即住 诚意出售', '恒大照母山山水城', '渝北照母山恒康路9号', '3室2厅1卫', '90.2㎡', '低层(共32层)', '2015年建造', '15530元/㎡']


In [10]:
import pymysql


conn = pymysql.connect(
    host="mysql",
    port=3306,
    user="student",
    password="123456",
    database="smartpark_lab",
    charset="utf8mb4",
    autocommit=False,   
)
cursor = conn.cursor()

cursor.execute("SELECT COUNT(*) FROM houseinfo")
before_count = cursor.fetchone()[0]
print("连接正常，入库前 houseinfo 条数：", before_count)

连接正常，入库前 houseinfo 条数： 498


In [11]:
INSERT_SQL = """INSERT INTO houseinfo
(title, house, address, struct, area, floor, maketime, price)
VALUES (%s, %s, %s, %s, %s, %s, %s, %s)"""

try:
    # 1. 读取CSV，检查表头，按8个字段的固定顺序整理记录
    with open("data/houseinfo.csv", "r", encoding="utf-8-sig", newline="") as f:
        reader = csv.DictReader(f)
        assert reader.fieldnames == CSV_HEADERS, f"表头不符：{reader.fieldnames}"
        records = [[row[h] for h in CSV_HEADERS] for row in reader]
    print("从CSV读入", len(records), "条记录")

    # 2. 按8个字段比较库内已有记录，只插入尚不存在的记录（避免重复运行时重复插入）
    cursor.execute("SELECT title, house, address, struct, area, floor, maketime, price FROM houseinfo")
    existing = set(tuple(r) for r in cursor.fetchall())
    new_records = [r for r in records if tuple(r) not in existing]
    print("库内已有", len(existing), "条，本次待插入", len(new_records), "条")

    # 3. 先用少量真实记录试写检查
    test_records = new_records[:5]
    if test_records:
        cursor.executemany(INSERT_SQL, test_records)  
        conn.commit()                                   
        print(f"试写 {len(test_records)} 条成功")

    # 4. 写入全部剩余记录
    cursor.execute("SELECT title, house, address, struct, area, floor, maketime, price FROM houseinfo")
    existing = set(tuple(r) for r in cursor.fetchall())
    remain = [r for r in records if tuple(r) not in existing]
    if remain:
        cursor.executemany(INSERT_SQL, remain)
        conn.commit()
    print(f"正式写入 {len(remain)} 条成功")

    # 5. 入库后条数
    cursor.execute("SELECT COUNT(*) FROM houseinfo")
    print("入库后 houseinfo 条数：", cursor.fetchone()[0])

except Exception as e:
    conn.rollback()   
    print("写入出错，已回滚：", e)

finally:
    cursor.close()    
    conn.close()
    print("游标和连接已关闭")

从CSV读入 498 条记录
库内已有 498 条，本次待插入 0 条
正式写入 0 条成功
入库后 houseinfo 条数： 498
游标和连接已关闭


In [12]:
# ---- 验收：重新连接查询条数和样本 ----
conn = pymysql.connect(host="mysql", port=3306, user="student", password="123456",
                       database="smartpark_lab", charset="utf8mb4", autocommit=False)
cursor = conn.cursor()

cursor.execute("SELECT COUNT(*) FROM houseinfo")
print("houseinfo 总条数：", cursor.fetchone()[0])

cursor.execute("SELECT title, house, address, struct, area, floor, maketime, price FROM houseinfo ORDER BY id LIMIT 5")
for r in cursor.fetchall():
    print(r)

cursor.close()
conn.close()

houseinfo 总条数： 498
('万科物业自营 南坪六公里金域翰林 三号线精装正三房带家具电器', '万科金域学府翰林菁华里', '南岸学府大道二塘路59号', '3室2厅1卫', '89.9㎡', '低层(共27层)', '2019年建造', '16465元/㎡')
('黄泥磅 两代一家 正规三房双卫 业主自住房 拎包入住', '两代一家', '渝北渝北黄泥磅紫荆路29号', '3室2厅2卫', '94.5㎡', '低层(共28层)', '2008年建造', '14815元/㎡')
('鸳鸯16年次新小区 大雅云居山刚需经典3房户型 随时看房诚售', '大雅云居山', '渝北鸳鸯龙展路59号', '3室2厅1卫', '88㎡', '低层(共24层)', '2016年建造', '16487元/㎡')
('中交中央公园 横厅精装大两房 出门都是公园', '中交中央公园悦颂', '渝北中央公园翠丹路4号', '2室2厅1卫', '91㎡', '低层(共25层)', '2016年建造', '20660元/㎡')
('富力现代广场 豪装大两房 户型方正 看房方便', '富力现代广场', '南岸弹子石弹子石新街59号', '2室2厅1卫', '85.5㎡', '低层(共30层)', '2012年建造', '16969元/㎡')


In [13]:
# ---- 抽取CSV中3条记录（第一条、跨页一条、最后一条），在数据库中逐字段核对 ----
sample_idx = [0, len(csv_rows) // 2, len(csv_rows) - 1]

conn = pymysql.connect(host="mysql", port=3306, user="student", password="123456",
                       database="smartpark_lab", charset="utf8mb4", autocommit=False)
cursor = conn.cursor()

for i in sample_idx:
    s = csv_rows[i]
    csv_row = tuple(s[h] for h in CSV_HEADERS)
    cursor.execute(
        "SELECT title, house, address, struct, area, floor, maketime, price "
        "FROM houseinfo WHERE title=%s AND house=%s LIMIT 1",
        (s["title"], s["house"]),
    )
    db_row = cursor.fetchone()
    if db_row is None:
        print(f"[第{i}条] 数据库中未找到：{s['title'][:20]}")
    elif tuple(db_row) == csv_row:
        print(f"[第{i}条] 8个字段与CSV完全一致：{s['title'][:20]}")
    else:
        print(f"[第{i}条] 不一致！")
        print("  CSV：", csv_row)
        print("  数据库：", tuple(db_row))

cursor.close()
conn.close()

[第0条] 8个字段与CSV完全一致：万科物业自营 南坪六公里金域翰林 三号线
[第249条] 8个字段与CSV完全一致：精装修洋房 位置安静 全新装修 随时看房
[第497条] 8个字段与CSV完全一致：恒大照母山 精装3房 内圈位置 户型方正
